In [3]:

import anndata as ad
import celltypist
from celltypist import models
import scanpy as sc
import numpy as np
import os
import os.path as op
import pandas as pd

/data_nfs/og86asub/netmap/netmap-evaluation/netmap/.pixi/envs/default/lib/python3.12/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


In [4]:
np.random.seed(12)


def object_with_markers(adata, barcodes, high_ui, marker_flat):
    
    barcodes['index'] = barcodes['index'].astype(str)
    barcodes = barcodes.set_index('index')
    adata = adata[barcodes.index].copy()
    adata.obs  = adata.obs.merge(barcodes, left_index=True, right_index = True)

    # mitochondrial genes, "MT-" for human, "Mt-" for mouse
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    # ribosomal genes
    adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
    # hemoglobin genes
    adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)
    sc.pp.filter_cells(adata, min_genes=500)
    sc.pp.filter_genes(adata, min_cells=50)
    adata = adata[adata.obs.pct_counts_mt<20]

    #sc.pp.scrublet(adata)

    #adata = adata[adata.obs.predicted_doublet == False]
    # Saving count data
    adata.layers["counts"] = adata.X.copy()

    # Normalizing to median total counts
    sc.pp.normalize_total(adata, target_sum = 10000)
    adata.layers['count_norm'] = adata.X.copy()
    # Logarithmize the data
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2500)

    sc.pp.pca(adata)
    sc.pp.neighbors(adata)
    sc.tl.leiden(adata)
    sc.tl.umap(adata)
    sc.tl.leiden(adata, resolution=0.35)



    flg = (adata.var.pct_dropout_by_counts<95) & (adata.var.highly_variable) & (~adata.var.index.isin(high_ui)) | (adata.var.index.isin(marker_flat))
    adata = adata[:,flg].copy()


    return adata




In [5]:
marker_dict = {
    # --- B cells and Plasma cells ---
    "B cell": ["CD19", "CD79A", "CD79B", "CD21", "IGHM", "IGHD", "IGHA1", "IGHA2", "IGHG1", "IGHG2", "IGHG3", "IGHG4", "IGHE", "IGKC", "IGLC2", "IGLC3", "MS4A1", "PAX5", "HLA-DR", "HLA-DQ"],
    "Transitional B cell": ["MME", "CD9", "CD38", "PAX5", "FCER2"],
    "Naive B cell": ["IL4R", "FCER2", "IGHM", "IGHD"],
    "Memory B cell": ["CD27", "AIM2", "IGHG1", "IGHG2", "IGHG3", "IGHG4", "IGHA1", "IGHA2", "IGHE"],
    "Effector B cell": ["ITGAX", "FCRL4", "FCRL5", "TBX21", "ZEB2", "PDCD1"],
    "Plasma cell": ["CD19", "PRDM1", "XBP1", "MZB1", "SLAMF7", "CD27", "CD38", "IGHM", "IGHA1", "IGHA2", "IGHG1", "IGHG2", "IGHG3", "IGHG4", "IGKC", "IGLC2", "IGLC3"],
    "Core naive B cell": ["IL4R", "FCER2", "IGHM", "IGHD"],
    "ISG+ naive B cell": ["STAT3", "STAT1", "IFI44L", "ISG15"],
    "Early memory B cell": ["CD27", "AIM2", "IGHA1", "IGHA2", "IGHG1", "IGHG2", "IGHG3", "IGHG4"],
    "Core memory B cell": ["CD27", "AIM2", "IGHG1", "IGHG2", "IGHG3", "IGHG4", "IGHA1", "IGHA2", "IGHE"],
    "Type 2 polarized memory B cell": ["IL4R", "FCER2", "COCH", "IGHG1", "IGHG4", "IGHE"],
    "CD95 memory B cell": ["FAS", "AIM2", "IGHA1", "IGHA2", "IGHG1", "IGHG2", "IGHG3", "IGHG4"],
    "Activated memory B cell": ["FOS", "CD69", "JUN", "MCL1", "MYC"],
    "CD27+ effector B cell": ["CD27"],
    "CD27- effector B cell": ["ITGAX", "TBX21", "ZEB2"],

    # --- T cell (shared Level 1) ---
    "T cell": ["TRAC", "TRDC", "CD3D", "CD3E", "CD3G"],

    # --- CD4 T, DN T, and Treg cells ---
    "Naive CD4 T cell": ["CD27", "CCR7", "SELL", "TCF7", "LEF1"],
    "Memory CD4 T cell": ["ITGB1"],
    "Treg": ["FOXP3", "IL2RA", "IKZF2", "RTKN2"],
    "DN T cell": ["TRAC"],
    "Proliferating T cell": ["MKI67"],
    "Core naive CD4 T cell": ["CD27", "CCR7", "SELL", "TCF7", "LEF1"],
    "SOX4+ naive CD4 T cell": ["SOX4"],
    "ISG+ naive CD4 T cell": ["MX1", "IFI44"],
    "CM CD4 T cell": ["CCR7", "SELL", "LEF1"],
    "GZMB- CD27- EM CD4 T cell": [],
    "GZMB- CD27+ EM CD4 T cell": ["CD27", "GZMK"],
    "KLRF1- GZMB+ CD27- memory CD4 T cell": ["GZMB", "CCL5"],
    "ISG+ memory CD4 T cell": ["MX1", "IFI44"],
    "Naive CD4 Treg": ["CD27", "CCR7", "SELL", "TCF7", "LEF1"],
    "Memory CD4 Treg": ["ITGB1"],
    "KLRB1+ memory CD4 Treg": ["KLRB1"],
    "GZMK+ memory CD4 Treg": ["GZMK"],
    "Memory CD8 Treg": ["CD8A"],
    "KLRB1+ memory CD8 Treg": ["KLRB1"],

    # --- CD8 T, gdT, and MAIT cells ---
    "Naive CD8 T cell": ["CD27", "CCR7", "SELL", "TCF7", "LEF1"],
    "Memory CD8 T cell": ["ITGB1", "GZMA", "GZMB", "GZMK", "TRAC"],
    "CD8aa": ["CD8A", "KLRC2", "IKZF2", "IL21R"],
    "MAIT": ["SLC4A10", "KLRB1"],
    "gdT": ["TRDC", "TRGC1", "TRGC2"],
    "Core naive CD8 T cell": ["CD27", "CCR7", "SELL", "TCF7", "LEF1"],
    "SOX4+ naive CD8 T cell": ["SOX4"],
    "ISG+ naive CD8 T cell": ["MX1", "IFI44"],
    "CM CD8 T cell": ["CCR7", "SELL", "LEF1"],
    "GZMK- CD27+ EM CD8 T cell": ["CD27"],
    "GZMK+ CD27+ EM CD8 T cell": ["CD27", "GZMK"],
    "KLRF1- GZMB+ CD27- EM CD8 T cell": ["GZMB", "CCL5"],
    "KLRF1+ GZMB+ CD27- EM CD8 T cell": ["KLRF1", "GZMB", "CCL5"],
    "ISG+ memory CD8 T cell": ["MX1", "IFI44"],
    "Naive Vd1 gdT": ["CCR7", "SELL", "LEF1"],
    "SOX4+ Vd1 gdT": ["SOX4"],
    "KLRF1- effector Vd1 gdT": ["TRDC", "TRDV1"],
    "KLRF1+ effector Vd1 gdT": ["TRDC", "TRDV1", "KLRF1"],
    "GZMB+ Vd2 gdT": ["TRDC", "TRDV2", "GZMB"],
    "GZMK+ Vd2 gdT": ["TRDC", "TRDV2", "GZMK"],
    "CD8 MAIT": ["CD8A"],
    "CD4 MAIT": ["CD4"],
    "ISG+ MAIT": ["MX1", "IFI44"],

    # --- NK cells and ILCs ---
    "NK cell": ["CD3E"],
    "ILC": [],
    "CD56bright NK cell": ["NCAM1"],
    "CD56dim NK cell": ["FCGR3A"],
    "Proliferating NK cell": ["MKI67"],
    "GZMK- CD56dim NK cell": [],
    "GZMK+ CD56dim NK cell": ["GZMK"],
    "Adaptive NK cell": ["KLRC2", "FCGR3A"],
    "ISG+ CD56dim NK cell": ["ISG15", "MX1", "MX2"],

    # --- Monocytes ---
    "Monocyte": ["FCN1", "CTSS"],
    "CD14 monocyte": ["CD14", "VCAN", "S100A8", "S100A9"],
    "CD16 monocyte": ["FCGR3A", "CDKN1C", "LST1"],
    "Intermediate monocyte": ["CD14", "FCGR3A", "HLA-DPA1", "HLA-DOA", "HLA-DRA", "CD74"],
    "Core CD14 monocyte": [],
    "IL1B+ CD14 monocyte": ["IL1B", "CCL3", "CXCL8"],
    "ISG+ CD14 monocyte": ["MX1", "IFI44L", "IFI6"],
    "Core CD16 monocyte": [],
    "C1Q+ CD16 monocyte": ["C1QA", "C1QB"],
    "ISG+ CD16 monocyte": ["MX1", "IFI44L", "IFI6"],

    # --- Dendritic cells ---
    "DC": ["CST3", "FLT3", "HLA-DPA1", "HLA-DRA", "CD74"],
    "ASDC": ["AXL", "SIGLEC6", "HAMP"],
    "cDC1": ["CLEC9A", "XCR1", "IDO1", "C1orf54"],
    "cDC2": ["CD1C", "FCN1", "PILRA"],
    "pDC": ["PTCRA", "SMIM5", "LAMP5", "IL3RA", "JCHAIN"],
    "CD14+ cDC2": ["CST3", "CD74", "HLA-DRA", "HLA-DPA1", "CD14", "S100A8", "S100A9", "VCAN", "CD163"],
    "HLA-DRhi cDC2": ["HLA-DPA1", "HLA-DRA", "CD1C"],
    "ISG+ cDC2": ["MX1", "IFI44L", "IFI6"],

    # --- Progenitors, Erythrocytes, and Platelets ---
    "Erythrocyte": ["HBA1", "HBA2", "HBB"],
    "Progenitor cell": ["SMIM24", "CD34"],
    "Platelet": ["PPBP", "TUBB1"],
    "CLP cell": [],
    "CMP cell": [],
    "BaEoMaP cell": [],
}

marker_flat = []
for m in marker_dict:
    current_list = marker_dict[m]
    marker_flat = marker_flat + current_list

In [6]:

pangalao = pd.read_csv('/data_nfs/og86asub/netmap/netmap-evaluation/data/markers/panglaodb_human.csv')
high_ui = pangalao[pangalao['UI']>0.1]['Official gene symbol'].values


In [7]:
experiment_id = 'bd-rhap-rep2'
output_dir = f'/data_nfs/og86asub/netmap/netmap-evaluation/data/blood/reprocessed/{experiment_id}'
os.makedirs(output_dir, exist_ok = True)
barcodes = pd.read_csv(op.join(output_dir, 'barcode_annotation.tsv'), sep = '\t')
adata = sc.read_10x_mtx('/data_nfs/og86asub/netmap/netmap-evaluation/data/blood/rhapsody/rhapsody_out/bd-rhap-rep2')
adata.var_names_make_unique()

adata = object_with_markers(adata,barcodes, high_ui, marker_flat)

adata.write_h5ad(op.join(output_dir, f"{experiment_id}_with_markers.h5ad"))


... storing 'celltype_semi_manual' as categorical
... storing 'feature_types' as categorical


In [8]:


experiment_id = 'bd-rhap-rep1'
output_dir = f'/data_nfs/og86asub/netmap/netmap-evaluation/data/blood/reprocessed/{experiment_id}'
os.makedirs(output_dir, exist_ok = True)
barcodes = pd.read_csv(op.join(output_dir, 'barcode_annotation.tsv'), sep = '\t')
adata2 = sc.read_10x_mtx('/data_nfs/og86asub/netmap/netmap-evaluation/data/blood/rhapsody/rhapsody_out/bd-rhap-rep1')
adata2.var_names_make_unique()

adata2 = object_with_markers(adata2, barcodes, high_ui, marker_flat)

adata2.write_h5ad(op.join(output_dir, f"{experiment_id}_with_markers.h5ad"))


... storing 'celltype_semi_manual' as categorical
... storing 'feature_types' as categorical


In [9]:
experiment_id = '10x-rep2-kallisto-cellbender'
output_dir = f'/data_nfs/og86asub/netmap/netmap-evaluation/data/blood/reprocessed/{experiment_id}'
os.makedirs(output_dir, exist_ok = True)
adata = sc.read_h5ad('/data_nfs/og86asub/netmap/netmap-evaluation/data/blood/SRR/kallisto/mtx_conversions/10x_rep2/10x_rep2_cellbender_filter_matrix.h5ad')

adata = adata[:, ~adata.var.gene_symbol.isna()]
adata.var = adata.var.reset_index()
adata.var = adata.var.set_index("gene_symbol")
adata.var.index = adata.var.index.astype(str)
adata.var_names_make_unique()
barcodes = pd.read_csv(op.join(output_dir, 'barcode_annotation.tsv'), sep = '\t')

adata = object_with_markers(adata, barcodes, high_ui, marker_flat)

adata.write_h5ad(op.join(output_dir, f"{experiment_id}_with_markers.h5ad"))

... storing 'celltype_semi_manual' as categorical


In [10]:
experiment_id = '10x-rep1-kallisto-cellbender'
output_dir = f'/data_nfs/og86asub/netmap/netmap-evaluation/data/blood/reprocessed/{experiment_id}'
os.makedirs(output_dir, exist_ok = True)

adata = sc.read_h5ad('/data_nfs/og86asub/netmap/netmap-evaluation/data/blood/SRR/kallisto/mtx_conversions/10x_rep1/10x_rep1_cellbender_filter_matrix.h5ad')

adata = adata[:, ~adata.var.gene_symbol.isna()]
adata.var = adata.var.reset_index()
adata.var = adata.var.set_index("gene_symbol")
adata.var.index = adata.var.index.astype(str)
adata.var_names_make_unique()
barcodes = pd.read_csv(op.join(output_dir, 'barcode_annotation.tsv'), sep = '\t')

adata = object_with_markers(adata, barcodes, high_ui, marker_flat)

adata.write_h5ad(op.join(output_dir, f"{experiment_id}_with_markers.h5ad"))

... storing 'celltype_semi_manual' as categorical


In [15]:
len(set(adata.var.index).intersection(set(marker_flat)))

96